# Create Jane and Aatos Erkko Foundation Awards (Jane ja Aatos Erkon säätiö)

Creates Jane and Aatos Erkko Foundation (JAES) research grants from two
first-party sources, merged in `scripts/local/jaes_to_s3.py`:

1. **jaes.fi "Donations granted"** year pages (2004-2025, one block per grant:
   grantee, institution, project title + duration, amount, optional abstract)
   and the site's own JSON endpoint `/wp-json/custom/v1/grants` that renders the
   current year (2026 and late-2025 decisions).
2. **research.fi** (national research information hub), to which JAES reports
   its science grants since 2023: grant number (`A866` form), leader with
   ORCID. 135 of its 137 JAES rows match a jaes.fi row on (year, amount); the
   2 unmatched 2023 rows are added from research.fi.

**526 research grants, 2005-2026, EUR 338.5M, 100% amount, 96% title,
97.5% named lead, 20% lead ORCID** (local run 2026-10-01).

**Scope filter (stated per batch rule).** JAES is a mixed foundation
(medicine, science and technology, plus arts/culture and societal causes):
- 2023+: the foundation's own category on each grant. Kept: Medicine,
  Technology, Other science. Dropped: Art and culture, (other) societal activity.
- 2004-2022 (no category published): kept when the grantee is a named
  researcher (academic/medical title, or a person with an institution and a
  research purpose) or an institution with an explicitly research purpose;
  12 rows kept by the institution rule (professorships, InstituteQ doctoral
  school, Turku total-body PET, Max Planck-UH centre, FIIA visiting
  professorship programmes, Aalto ARTS summer school) are marked
  `scope_rule = 'institution_research'` in the raw table for review.
  Dropped: arts, music, festivals, restorations, hospital equipment,
  school scholarships, fundraising campaigns (337 of 861 site rows).
- Co-funded call totals ('JAES and Technology Industries of Finland
  Centennial', 'Future Makers') are programme totals, not grants: dropped.
- 2006's page uses an older layout with no grant blocks (its list is
  institutional/cultural donations); nothing is taken from it.

**Field mapping:** `display_name` = project title, else composed
'Jane and Aatos Erkko Foundation grant: <grantee> (<year>)' (19 rows whose
listing shows only grantee + institution). `start_year` = grant year;
`start_date`/`end_date` NULL (the site gives the year and a duration only).
`funder_scheme` = the foundation's category (2023+) or 'Research grant'.

**`funder_award_id` (2.1.1):** the research.fi grant number where matched
(137 rows, `A\d{3}`; no shells use it yet), else synthetic
`JAES-<year>-<lead family>-<lead given>`. Citing works write JAES's 6-digit
application numbers (e.g. `200063`; 277 of the 660 acknowledgement shell
rows), which neither source publishes, so those shells cannot be collapsed.

**Funder details (Path A, F4320* Crossref-registered):**
- funder_id: `4320322282`
- display_name / ror_id / doi: from `openalex.funders.funders` (ror 03vxy9y38, doi 10.13039/501100004012)

**Priority:** `525` (direct funder ingest; higher wins under the 2026-06-20 DESC dedup).

## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.jaes_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/jaes/jaes_projects.parquet`;

In [ ]:
%sql
SELECT COUNT(*) as total_grants FROM openalex.awards.jaes_raw;

In [ ]:
%sql
DESCRIBE openalex.awards.jaes_raw;

In [ ]:
%sql
SELECT scope_rule, source, COUNT(*) AS n FROM openalex.awards.jaes_raw GROUP BY 1, 2 ORDER BY n DESC;

## Step 1.6: Funder existence check (Path A)

In [ ]:
%sql
SELECT assert_true(COUNT(*) = 1, 'F4320322282 must have exactly one openalex.funders.funders row') AS funder_ok,
       MAX(display_name) AS display_name, MAX(ror_id) AS ror_id, MAX(doi) AS doi
FROM openalex.funders.funders
WHERE funder_id = 4320322282;

## Step 2: Create JAES Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.jaes_awards
USING delta
AS
WITH
src_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.funders.funders
    WHERE funder_id = 4320322282  -- Jane ja Aatos Erkon Säätiö
),

people AS (
    -- struct field order MUST match openalex_awards_raw: given, family, orcid, role_start, affiliation.
    SELECT
        funder_award_id,
        TRANSFORM(
            FILTER(from_json(people, 'array<struct<name:string,given_name:string,family_name:string,orcid:string>>'),
                   p -> p.family_name IS NOT NULL),
            p -> named_struct(
                'given_name', NULLIF(TRIM(p.given_name), ''),
                'family_name', NULLIF(TRIM(p.family_name), ''),
                'orcid', CASE WHEN p.orcid RLIKE '^[0-9]{4}-[0-9]{4}-[0-9]{4}-[0-9]{3}[0-9X]$'
                              THEN CONCAT('https://orcid.org/', p.orcid) END,
                'role_start', CAST(NULL AS DATE),
                'affiliation', named_struct(
                    'name', NULLIF(TRIM(institution), ''),
                    'country', CAST(NULL AS STRING),
                    'ids', CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>)
                )
            )) AS investigators
    FROM openalex.awards.jaes_raw
)

SELECT
    abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000 as id,
    COALESCE(NULLIF(TRIM(g.title), ''),
             CONCAT('Jane and Aatos Erkko Foundation grant: ', COALESCE(g.grantee, g.lead_name), ' (', g.grant_year, ')')) as display_name,
    NULLIF(TRIM(g.abstract), '') as description,
    f.funder_id,
    TRIM(g.funder_award_id) as funder_award_id,
    TRY_CAST(g.amount AS DOUBLE) as amount,
    g.currency as currency,
    struct(
        CONCAT('https://openalex.org/F', f.funder_id) as id,
        f.display_name,
        f.ror_id,
        f.doi
    ) as funder,
    'research' as funding_type,
    CASE
        WHEN lower(g.category) LIKE 'medicin%' THEN 'Medicine'
        WHEN lower(g.category) = 'technology' THEN 'Technology'
        WHEN lower(g.category) LIKE 'other%science%' THEN 'Other science'
        ELSE 'Research grant'
    END as funder_scheme,
    'jaes_donations_granted' as provenance,
    CAST(NULL AS DATE) as start_date,
    CAST(NULL AS DATE) as end_date,
    TRY_CAST(g.grant_year AS INT) as start_year,
    CAST(NULL AS INT) as end_year,
    -- try_element_at: safe on empty arrays (2.3)
    try_element_at(p.investigators, 1) as lead_investigator,
    try_element_at(p.investigators, 2) as co_lead_investigator,
    CASE WHEN size(p.investigators) > 0 THEN p.investigators END as investigators,
    g.landing_page_url as landing_page_url,
    CAST(NULL AS STRING) as doi,
    concat('https://api.openalex.org/works?filter=awards.id:G',
           abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000) as works_api_url,
    current_timestamp() as created_date,
    current_timestamp() as updated_date
FROM openalex.awards.jaes_raw g
JOIN people p ON p.funder_award_id = g.funder_award_id
CROSS JOIN src_funder f
WHERE g.funder_award_id IS NOT NULL AND TRIM(g.funder_award_id) != '';

## Shape check (2.1.1)
research.fi grant numbers (`A\d{3,4}`) or the synthetic `JAES-<year>-...` key only.

In [ ]:
%sql
SELECT assert_true(COUNT(*) = 0, 'unexpected funder_award_id shape') AS shape_ok
FROM openalex.awards.jaes_awards
WHERE NOT (funder_award_id RLIKE '^A[0-9]{3,4}$' OR funder_award_id RLIKE '^JAES-(19|20)[0-9]{2}-[a-z0-9-]+$');

In [ ]:
%sql
SELECT assert_true(COUNT(*) = COUNT(DISTINCT id), 'duplicate award ids') AS unique_ok, COUNT(*) AS n
FROM openalex.awards.jaes_awards;

## Step 3: Insert into openalex_awards_raw

In [ ]:
%sql
-- Remove previous data for this source before inserting fresh data.
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'jaes_donations_granted' AND priority = 525;

-- Priority 525: direct-from-funder ingest of JAES's own grant listings (+ its research.fi reports).
-- Higher wins under the 2026-06-20 DESC dedup (oxjob #500).
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id, amount, currency,
    funder, funding_type, funder_scheme, provenance, start_date, end_date,
    start_year, end_year, lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url, created_date, updated_date,
    525 as priority
FROM openalex.awards.jaes_awards;

## Verification Queries

In [ ]:
%sql
SELECT id, display_name, funder_award_id, funder_scheme, amount, start_year,
       lead_investigator.given_name, lead_investigator.family_name, lead_investigator.orcid,
       lead_investigator.affiliation.name
FROM openalex.awards.jaes_awards LIMIT 20;

In [ ]:
%sql
SELECT funder_scheme, COUNT(*) as cnt, ROUND(SUM(amount), 0) as total_eur
FROM openalex.awards.jaes_awards GROUP BY 1 ORDER BY cnt DESC;

In [ ]:
%sql
SELECT start_year, COUNT(*) as cnt, ROUND(SUM(amount), 0) as total_eur
FROM openalex.awards.jaes_awards
WHERE start_year IS NOT NULL GROUP BY start_year ORDER BY start_year;

In [ ]:
%sql
-- 6.4a frequency check: PI top-20 must be a real long-tail.
SELECT lead_investigator.given_name AS given, lead_investigator.family_name AS family, COUNT(*) AS n
FROM openalex.awards.jaes_awards
GROUP BY 1, 2 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
-- 6.3 / 6.7 coverage.
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(description) as has_description,
    COUNT(amount) as has_amount,
    COUNT(DISTINCT currency) as distinct_currencies,
    ROUND(MIN(amount), 0) as min_amt,
    ROUND(MAX(amount), 0) as max_amt,
    ROUND(AVG(amount), 0) as avg_amt,
    COUNT(start_year) as has_start_year,
    COUNT(lead_investigator) as has_pi,
    COUNT(lead_investigator.orcid) as has_orcid
FROM openalex.awards.jaes_awards;

In [ ]:
%sql
-- Confirm rows reached the shared raw table at the assigned priority (6.8).
SELECT provenance, priority, COUNT(*) as n
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'jaes_donations_granted'
GROUP BY provenance, priority;